In [6]:
import requests
import pandas as pd

DGIDB_URL = "https://dgidb.org/api/graphql"

query = """
query BCL2PMIDs {
  genes(names: ["BCL2","EGFR"]) {
    nodes {
      interactions {
        drug {
          name
          conceptId
        }
        gene {
          name
            }
        publications {
          pmid
        }
        sources {
          sourceDbName
        }
      }
    }
  }
}
"""

r = requests.post(DGIDB_URL, json={"query": query})
r.raise_for_status()
data = r.json()

if "errors" in data:
    raise RuntimeError(data["errors"])

rows = []

for gene in data["data"]["genes"]["nodes"]:
    for interaction in gene.get("interactions", []):
        drug = interaction.get("drug") or {}
        sources = interaction.get("sources") or []
        gene = interaction.get("gene") or {}
        source_names = sorted({
            s.get("sourceDbName")
            for s in sources
            if s.get("sourceDbName")
        })

        for pub in interaction.get("publications", []):
            pmid = pub.get("pmid")
            if pmid:
                rows.append({
                    "pmid": str(pmid),
                    "drug_name": drug.get("name"),
                    "gene_name": gene.get("name"),
                    "drug_concept_id": drug.get("conceptId"),
                    "sources": "; ".join(source_names),
                })

pmid_df = (
    pd.DataFrame(rows)
    .drop_duplicates(subset=["pmid"])
    .head(100)
    .reset_index(drop=True)
)

print(f"Found {len(pmid_df)} unique PMIDs associated with BCL2 interactions")
pmid_df

Found 100 unique PMIDs associated with BCL2 interactions


,pmid,drug_name,gene_name,drug_concept_id,sources
0,14977850,OXALIPLATIN,BCL2,rxcui:32592,ClearityFoundationBiomarkers; NCI
1,16684279,METHYLPREDNISOLONE,BCL2,rxcui:6902,NCI
2,16141680,METHYLPREDNISOLONE,BCL2,rxcui:6902,NCI
3,24749893,NAVITOCLAX,BCL2,ncit:C64776,ChEMBL; DTC; TALC; TTD
4,12576461,EPIDERMAL GROWTH FACTOR RECEPTOR TYROSINE KINA...,BCL2,ncit:C2167,NCI
...,...,...,...,...,...
95,16204070,ERLOTINIB,EGFR,rxcui:337525,CGI; CIViC; COSMIC; CancerCommons; ClearityFou...
96,19147750,ERLOTINIB,EGFR,rxcui:337525,CGI; CIViC; COSMIC; CancerCommons; ClearityFou...
97,15710947,ERLOTINIB,EGFR,rxcui:337525,CGI; CIViC; COSMIC; CancerCommons; ClearityFou...
98,23945392,ERLOTINIB,EGFR,rxcui:337525,CGI; CIViC; COSMIC; CancerCommons; ClearityFou...


In [7]:
import requests
import xml.etree.ElementTree as ET
from time import sleep

def fetch_pubmed_abstracts(pmids, batch_size=100, email=None):
    """
    Fetch complete PubMed abstract text for PMIDs using NCBI E-utilities.
    Returns dict: {pmid: abstract_text}
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    abstracts = {}

    pmids = [str(pmid) for pmid in pmids if pd.notna(pmid)]

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]

        params = {
            "db": "pubmed",
            "id": ",".join(batch),
            "retmode": "xml",
        }

        if email:
            params["email"] = email

        r = requests.get(base_url, params=params)
        r.raise_for_status()

        root = ET.fromstring(r.content)

        for article in root.findall(".//PubmedArticle"):
            pmid = article.findtext(".//PMID")

            abstract_parts = []
            for abstract_text in article.findall(".//Abstract/AbstractText"):
                label = abstract_text.attrib.get("Label")
                text = "".join(abstract_text.itertext()).strip()

                if text:
                    if label:
                        abstract_parts.append(f"{label}: {text}")
                    else:
                        abstract_parts.append(text)

            abstracts[str(pmid)] = "\n".join(abstract_parts) if abstract_parts else None

        sleep(0.34)  # stay under NCBI's 3 requests/second limit

    return abstracts


abstract_map = fetch_pubmed_abstracts(
    pmid_df["pmid"],
    email="matthew.cannon2@nationwidechildrens.org"
)

pmid_df["context"] = pmid_df["pmid"].astype(str).map(abstract_map)

pmid_df.head()

,pmid,drug_name,gene_name,drug_concept_id,sources,context
0,14977850,OXALIPLATIN,BCL2,rxcui:32592,ClearityFoundationBiomarkers; NCI,PURPOSE: Overexpression of antiapoptotic Bcl-2...
1,16684279,METHYLPREDNISOLONE,BCL2,rxcui:6902,NCI,OBJECTIVE: The decrease of physiological apopt...
2,16141680,METHYLPREDNISOLONE,BCL2,rxcui:6902,NCI,"Cardiac injury, occurred after traumatic brain..."
3,24749893,NAVITOCLAX,BCL2,ncit:C64776,ChEMBL; DTC; TALC; TTD,"Mcl-1, an antiapoptotic member of the Bcl-2 fa..."
4,12576461,EPIDERMAL GROWTH FACTOR RECEPTOR TYROSINE KINA...,BCL2,ncit:C2167,NCI,PURPOSE: This study investigated whether the f...


In [8]:
pmid_df.to_excel('2026-06-08-test1.xlsx')